<a href="https://colab.research.google.com/github/BrandonTatani/MLProject/blob/master/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [2]:
from datasets import Dataset

In [3]:
DATA_DIR = 'data'

MODEL_NAME = "facebook/bart-base"
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 256
SEED = 42



In [4]:
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Loading model...')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

In [14]:
class Preprocessor:
    def __init__(self):

        # loading tokenized dataset from cache if available
        print("Loading cached dataset...")
        self.tokenized_paths = {
        'train' : '/content/drive/MyDrive/data/train_tokenized',
        'validation' : '/content/drive/MyDrive/data/validation_tokenized',
        'test' : '/content/drive/MyDrive/data/test_tokenized',
        }

        self.splits = {
            name: Dataset.load_from_disk(path)
            for name, path in self.tokenized_paths.items()
        }

        print(self.splits)



    def train(self):
      """
          :return: tokenized train split
      """
      return self.splits['train']

    def eval(self):
        """
        :return: tokenized validation split
        """
        return self.splits['validation']

    def test(self):
        """
        :return: tokenized test split
        """
        return self.splits['test']


In [7]:
!pip install evaluate

In [8]:
from transformers import Trainer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import evaluate

In [10]:
!pip install rouge_score

In [11]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decodifica i token in testo leggibile
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Calcolo ROUGE
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_aggregator = False)

    return result

In [18]:
preprocessed = Preprocessor()

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)

training_args = Seq2SeqTrainingArguments(
    output_dir = "output",
    save_strategy = "epoch",
    eval_strategy="steps",
    eval_steps=300,
    fp16 = True,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size = 2,
    gradient_accumulation_steps = 8,
    learning_rate = 5e-5,
    num_train_epochs = 5, # maximum number of epochs
    weight_decay = 0.01,
    save_total_limit = 2,
    predict_with_generate = True,
    logging_dir= "logs",
    logging_steps = 300,
    overwrite_output_dir=True,
    dataloader_num_workers = 2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = preprocessed.train(),
    eval_dataset = preprocessed.eval(),
    data_collator = collator,
    tokenizer = tokenizer,
    compute_metrics = compute_metrics
)

trainer.train()
results = trainer.evaluate(preprocessed.test())
print(results)

In [21]:
from matplotlib import pyplot as plt
from statistics import mean, median
print(results.keys())
print(mean(results['eval_rouge1']))
print(mean(results['eval_rouge2']))
print(mean(results['eval_rougeL']))
print(median(results['eval_rouge1']))
print(median(results['eval_rouge2']))
print(median(results['eval_rougeL']))

plt.hist(results['eval_rouge1'], bins=20, label='rouge1', alpha=0.5)
plt.hist(results['eval_rouge2'], bins=20, label='rouge2', alpha=0.5)
plt.hist(results['eval_rougeL'], bins=20, label='rougeL', alpha=0.5)
plt.legend()
plt.show()

In [22]:
import torch
sample = preprocessed.test()[0]
source_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
print("_________________SOURCE___________________")
print(source_text)


target_text = tokenizer.decode(sample['labels'], skip_special_tokens=True)
print("__________________TARGET____________________")
print(target_text)


inputs = {k: torch.tensor([v]).to(trainer.model.device)
          for k, v in sample.items() if k == "input_ids"}

generated_ids = trainer.model.generate(
    inputs["input_ids"],
    max_length=MAX_TARGET_LENGTH,
    num_beams=4,
)

pred_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("__________________PREDICTED____________________")
print(pred_text)

In [45]:
import json
with open('results.json', 'w') as f:
  json.dump(results, f, indent=4)

with open('comparison_text.txt', 'w') as f:
  f.write(f'Source: {source_text}\n')
  f.write(f'Target: {target_text}\n')
  f.write(f'Predicted: {pred_text}\n')

trainer.save_model('trained_model')
tokenizer.save_pretrained('trained_model')

In [46]:
!cp -r trained_model drive/MyDrive/

In [47]:
!cp -r results.json drive/MyDrive/
!cp -r comparison_text.txt drive/MyDrive/
!cp -r output drive/MyDrive/
!cp -r logs drive/MyDrive/

In [75]:
import matplotlib.pyplot as plt
import pandas as pd

# log_history stores training/eval losses
logs = trainer.state.log_history
print(logs)

# convert to DataFrame
df = pd.DataFrame(logs)

# separate train/eval losses
t_loss = df.dropna(subset=["loss"], axis=0)
print(t_loss[["loss", "step"]])
e_loss = df.dropna(subset=["eval_loss"], axis=0)

# plot
plt.figure(figsize=(10,6))
plt.plot(t_loss["step"], t_loss["loss"], 'r-', label="Training Loss")
if "eval_loss" in df:
    plt.plot(e_loss["step"], e_loss["eval_loss"], label="Eval Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Training & Evaluation Loss")
plt.show()


In [13]:
from google.colab import drive
drive.mount('/content/drive')